# Bloque 7 — Grafos e IA
## Notebook 01: Agente investigador sobre la red

---

En el notebook anterior construimos el grafo y calculamos métricas. Ahora añadimos un agente de IA encima: en vez de consultar el grafo manualmente, le hacemos **preguntas en lenguaje natural** y el agente decide qué herramientas usar para responderlas.

### Pipeline

```
 Pregunta en texto
       ↓
 [Agente Analista] — qwen2.5:14b
   ├── contactos_directos(numero)
   ├── contactos_comunes(numero1, numero2)
   ├── comunidad_de(numero)
   ├── nodos_puente(comunidad_a, comunidad_b)
   └── perfil_llamadas(numero)
       ↓
 [Agente Redactor] — deepseek-r1:14b
       ↓
  Informe estructurado
```

> **Prerrequisito**: ejecutar `00_construccion_red.ipynb` antes de este notebook para generar `data/grafo.pkl`, `data/nodos.csv` y `data/llamadas.csv`.

In [ ]:
# ─── CARGAR EL GRAFO Y LOS DATOS ─────────────────────────────────────────────
import pickle
import pandas as pd
import igraph as ig
from pathlib import Path

DATA = Path('data')

with open(DATA / 'grafo.pkl', 'rb') as f:
    G = pickle.load(f)

df_nodos   = pd.read_csv(DATA / 'nodos.csv')
df_llamadas = pd.read_csv(DATA / 'llamadas.csv')
df_llamadas['timestamp'] = pd.to_datetime(df_llamadas['timestamp'])

# Índice nombre → id de vértice
NOMBRE_IDX = {v['nombre']: v.index for v in G.vs}

print(f'Grafo cargado: {G.vcount():,} nodos | {G.ecount():,} aristas')
print(f'Comunidades  : {df_nodos["comunidad"].nunique()}')
print(f'Llamadas     : {len(df_llamadas):,}')
print(f'\nTeléfonos core:')
for n, g_id, com in df_nodos[df_nodos["tipo"].str.startswith("Grupo")][["nombre", "tipo", "comunidad"]].values:
    print(f'  {n}  {g_id:<10}  comunidad {com}')

In [ ]:
# ─── HERRAMIENTAS DE CONSULTA SOBRE EL GRAFO ─────────────────────────────────
from crewai.tools import tool

@tool
def contactos_directos(numero: str) -> str:
    """Lista los números con los que ha llamado este teléfono y cuántas veces, ordenados por frecuencia."""
    if numero not in NOMBRE_IDX:
        return f"Número {numero} no encontrado en el grafo."
    idx = NOMBRE_IDX[numero]
    vecinos = G.neighbors(idx)
    resultado = []
    for v in vecinos:
        eid = G.get_eid(idx, v)
        resultado.append((G.vs[v]['nombre'], G.es[eid]['peso']))
    resultado.sort(key=lambda x: -x[1])
    lineas = [f'  {n}: {int(p)} llamadas' for n, p in resultado[:20]]
    return f"Contactos de {numero} (top {len(lineas)}):\n" + "\n".join(lineas)


@tool
def contactos_comunes(numero1: str, numero2: str) -> str:
    """Devuelve los teléfonos con los que han llamado ambos números a la vez (contactos compartidos)."""
    for n in [numero1, numero2]:
        if n not in NOMBRE_IDX:
            return f"Número {n} no encontrado."
    vec1 = {G.vs[v]['nombre'] for v in G.neighbors(NOMBRE_IDX[numero1])}
    vec2 = {G.vs[v]['nombre'] for v in G.neighbors(NOMBRE_IDX[numero2])}
    comunes = vec1 & vec2 - {numero1, numero2}
    if not comunes:
        return f"No hay contactos comunes entre {numero1} y {numero2}."
    return (f"Contactos comunes entre {numero1} y {numero2} ({len(comunes)} en total):\n" +
            "\n".join(f"  {c}" for c in sorted(comunes)[:20]))


@tool
def comunidad_de(numero: str) -> str:
    """Devuelve la comunidad Leiden a la que pertenece un número y cuántos miembros tiene esa comunidad."""
    if numero not in NOMBRE_IDX:
        return f"Número {numero} no encontrado."
    com = G.vs[NOMBRE_IDX[numero]]['comunidad']
    miembros = [v['nombre'] for v in G.vs if v['comunidad'] == com]
    muestra = ', '.join(miembros[:5])
    return (f"{numero} → comunidad {com} ({len(miembros)} miembros).\n"
            f"Muestra: {muestra}{'...' if len(miembros) > 5 else ''}")


@tool
def nodos_puente(comunidad_a: int, comunidad_b: int) -> str:
    """Encuentra nodos que tienen contactos en dos comunidades distintas (brokers o intermediarios)."""
    brokers = []
    for v in G.vs:
        vecinos_coms = {G.vs[n]['comunidad'] for n in G.neighbors(v.index)}
        if comunidad_a in vecinos_coms and comunidad_b in vecinos_coms:
            brokers.append((v['nombre'], v['comunidad'], G.degree(v.index)))
    if not brokers:
        return f"No hay brokers entre comunidad {comunidad_a} y comunidad {comunidad_b}."
    brokers.sort(key=lambda x: -x[2])
    lineas = [f"  {n} (comunidad {c}, grado {d})" for n, c, d in brokers[:10]]
    return f"Brokers entre comunidades {comunidad_a} y {comunidad_b}:\n" + "\n".join(lineas)


@tool
def perfil_llamadas(numero: str) -> str:
    """Estadísticas de llamadas de un número: total, duración media, hora pico, ratio saliente/entrante."""
    mask = (df_llamadas['origen'] == numero) | (df_llamadas['destino'] == numero)
    sub = df_llamadas[mask]
    if sub.empty:
        return f"No hay llamadas para {numero}."
    hora_pico = sub['timestamp'].dt.hour.value_counts().idxmax()
    salientes = (sub['origen'] == numero).sum()
    entrantes = (sub['destino'] == numero).sum()
    return (
        f"Perfil de {numero}:\n"
        f"  Total llamadas   : {len(sub):,}\n"
        f"  Duración media   : {sub['duracion_s'].mean():.0f}s\n"
        f"  Duración total   : {sub['duracion_s'].sum() / 3600:.1f}h\n"
        f"  Hora pico        : {hora_pico:02d}:00h\n"
        f"  Salientes        : {salientes}\n"
        f"  Entrantes        : {entrantes}"
    )

print('Herramientas definidas: contactos_directos, contactos_comunes, comunidad_de, nodos_puente, perfil_llamadas')

In [ ]:
# ─── CONFIGURAR AGENTES ───────────────────────────────────────────────────────
from crewai import Agent, Task, Crew, Process, LLM

OLLAMA_URL = 'http://localhost:11434'
llm_analista = LLM(model='ollama/qwen2.5:14b', base_url=OLLAMA_URL)
llm_redactor = LLM(model='ollama/deepseek-r1:14b', base_url=OLLAMA_URL)

HERRAMIENTAS = [
    contactos_directos, contactos_comunes,
    comunidad_de, nodos_puente, perfil_llamadas,
]

agente_analista = Agent(
    role='Analista de redes de comunicación',
    goal='Investigar patrones de llamadas usando las herramientas de consulta sobre el grafo y extraer hechos concretos.',
    backstory=(
        'Eres un analista especializado en redes de comunicación. '
        'Usas las herramientas disponibles para consultar el grafo y presentas '
        'tus hallazgos de forma estructurada y factual.'
    ),
    tools=HERRAMIENTAS,
    llm=llm_analista,
    verbose=True,
)

agente_redactor = Agent(
    role='Redactor de informes de inteligencia',
    goal='Sintetizar el análisis del analista en un informe claro, estructurado y accionable.',
    backstory=(
        'Redactas informes de inteligencia a partir del análisis del investigador. '
        'Eres preciso, objetivo y usas formato estructurado en markdown.'
    ),
    llm=llm_redactor,
    verbose=True,
)

print('Agentes listos: analista (qwen2.5:14b) + redactor (deepseek-r1:14b)')

In [ ]:
# ─── HELPER: lanzar crew sin bloquear el event loop de Jupyter ────────────────
import concurrent.futures

def investigar(pregunta: str, sujeto: str = '') -> str:
    """
    Lanza el crew analista+redactor con la pregunta dada.
    sujeto: contexto adicional (número o números de interés).
    """
    contexto = f"Sujeto(s) de interés: {sujeto}\n\n" if sujeto else ''

    tarea_analisis = Task(
        description=(
            f"{contexto}Pregunta de investigación:\n{pregunta}\n\n"
            "Usa las herramientas disponibles para obtener datos concretos del grafo. "
            "Presenta los hallazgos como una lista de hechos verificados."
        ),
        expected_output='Lista estructurada de hechos: comunidad, contactos clave, rol en la red, métricas relevantes.',
        agent=agente_analista,
    )

    tarea_informe = Task(
        description=(
            "Con los hallazgos del analista, redacta un informe de inteligencia en markdown. "
            "Incluye: Resumen Ejecutivo, Posición en la Red, Contactos Clave, Conclusiones."
        ),
        expected_output='Informe en markdown con secciones: ## Resumen Ejecutivo, ## Posición en la Red, ## Contactos Clave, ## Conclusiones.',
        agent=agente_redactor,
        context=[tarea_analisis],
    )

    crew = Crew(
        agents=[agente_analista, agente_redactor],
        tasks=[tarea_analisis, tarea_informe],
        process=Process.sequential,
        verbose=False,
    )

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        resultado = ex.submit(crew.kickoff).result(timeout=600)

    return str(resultado)

print('Helper investigar() listo')

## Demo 1 — Investigar un número sujeto

Pedimos al crew que analice el teléfono `+34600000001` (grupo A): su comunidad, contactos clave y rol en la red.

In [ ]:
# ─── DEMO 1: investigar un número ────────────────────────────────────────────
SUJETO = '+34600000001'

informe1 = investigar(
    pregunta=(
        f"Investiga el teléfono {SUJETO}. "
        "Determina: (1) a qué comunidad pertenece, "
        "(2) sus 10 contactos más frecuentes y qué tipo de nodos son, "
        "(3) su perfil de llamadas (horario pico, duración media), "
        "(4) si actúa como puente entre comunidades."
    ),
    sujeto=SUJETO,
)

print(informe1)

## Demo 2 — ¿Quién conecta dos grupos?

Preguntamos qué nodos actúan de puente entre la comunidad de los grupos A y B, y qué tienen en común dos números sujeto de grupos distintos.

In [ ]:
# ─── DEMO 2: brokers entre grupos ────────────────────────────────────────────
# Averiguar las comunidades de dos sujetos de grupos distintos
com_A = G.vs[NOMBRE_IDX['+34600000001']]['comunidad']
com_B = G.vs[NOMBRE_IDX['+34600000005']]['comunidad']

informe2 = investigar(
    pregunta=(
        f"Identifica qué nodos actúan de puente entre la comunidad {com_A} y la comunidad {com_B}. "
        f"Analiza también qué contactos comparten +34600000001 y +34600000005. "
        "Explica el papel de estos brokers en la red."
    ),
    sujeto=f'+34600000001 (comunidad {com_A}) y +34600000005 (comunidad {com_B})',
)

print(informe2)

## Demo 3 — ¿Qué tienen en común tres números?

Dados tres números de grupos distintos, el crew busca si hay contactos comunes y qué dice eso sobre sus relaciones.

In [ ]:
# ─── DEMO 3: red de tres números ─────────────────────────────────────────────
TRES = ['+34600000002', '+34600000006', '+34600000009']

informe3 = investigar(
    pregunta=(
        f"Analiza los tres teléfonos: {', '.join(TRES)}. "
        "Para cada par, identifica si tienen contactos comunes. "
        "¿Pertenecen a la misma comunidad o a comunidades distintas? "
        "¿Existe algún intermediario que los conecte a todos?"
    ),
    sujeto=', '.join(TRES),
)

print(informe3)